In [0]:
!lsb_release -a

In [0]:
%fs ls

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
import requests
import zipfile
import os
import pandas as pd
from datetime import datetime

In [0]:
# Setting these variables for jobs purpose
# {{current_date}}

dbutils.widgets.text("current_date", "2026-07-01", "Current Date (YYYY-MM-DD)")
dbutils.widgets.dropdown("mode", "overwrite", ["overwrite", "append"], "Write Mode")

current_date = dbutils.widgets.get("current_date") #YYYY-MM-DD
write_mode = dbutils.widgets.get("mode") #YYYY-MM-DD

run_date_dt = datetime.strptime(current_date, '%Y-%m-%d')

year_month = run_date_dt.strftime('%Y_%m') #YYYY_MM

In [0]:
source_data = {
    "On_Time_performance_Report_URL" : f"https://transtats.bts.gov/PREZIP/On_Time_Reporting_Carrier_On_Time_Performance_1987_present_{year_month}.zip",

    "Airports_Maps": "https://www.transtats.bts.gov/Download_Lookup.asp?Y11x72=Y_NVecbeg",
    
    "Carriers_Maps": "https://www.transtats.bts.gov/Download_Lookup.asp?Y11x72=Y_haVdhR_PNeeVRef"
}

In [0]:
def download_zip(url, output_path):
    response = requests.get(url, stream=True)
    response.raise_for_status()  # Raise error for bad status
    with open(output_path, 'wb') as f:
        for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk)
    print(f"Download complete; File in path {output_path}")

In [0]:
def extract_zip(zip_path, extract_to):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_to)
    print(f"Extraction complete; File in path {extract_to}")

In [0]:
def find_csv_file(directory):
    for file in os.listdir(directory):
        if file.endswith('.csv'):
            return os.path.join(directory, file)
    return None

In [0]:
data_path = "/Volumes/workspace/default/sample_dataset"

In [0]:
for source, url in source_data.items():
    print(f"Processing {source}...")
    ZIP_URL = url
    ZIP_FILE = data_path + "/" + source + ".zip"; os.makedirs(os.path.dirname(ZIP_FILE), exist_ok=True)
    EXTRACT_DIR = data_path + "/" + source + "/"

    download_zip(ZIP_URL, ZIP_FILE)
    try:
        extract_zip(ZIP_FILE, EXTRACT_DIR)
        csv_file = find_csv_file(EXTRACT_DIR)

    # print(csv_file)
    except zipfile.BadZipFile:
        print(f"Error: File is not a zip file.", e, EXTRACT_DIR)
        csv_file = ZIP_FILE
    if csv_file:
        print(f"reading  file from {csv_file}")
        pyspark_df = spark.read.csv(csv_file, header=True, inferSchema=True)
        pyspark_df.write.mode(write_mode).saveAsTable(f"workspace.default.{source}")
    else:
        print("No CSV file found in the extracted ZIP.")

In [0]:
spark.table('workspace.default.on_time_performance_report_url').display()

In [0]:
 %sql 
 select * from workspace.default.carriers_maps